In [12]:
from pathlib import Path
import sys
import json
ROOT = Path.cwd().parent
DATA = ROOT / 'data'
sys.path.append(str(ROOT))
from src.chunks import read_pages
from src.bm25 import BM25Retriever,tokenize
from src.chroma import load_chroma
from src.retrieval import build_hybride_retriever
from src.evaluation import load_eval, comparer_poids
from pprint import pp
from dotenv import load_dotenv
load_dotenv()   # lit .env et remplit os.environ
from langchain_mistralai import ChatMistralAI

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.rate_limiters import InMemoryRateLimiter

# Retriever

In [2]:
chunks = read_pages(DATA / 'processed/chunks.jsonl')
K_RETRIEVER = 10        # par bm25 / chroma
K_EVAL = 5     # hit@5
weights_rrf = [.3, .7]

In [3]:
bm25_retriever = BM25Retriever.from_documents(chunks, tokenize, k=K_RETRIEVER)

chroma_retriever = load_chroma(collection_name='rapport_v0',
                               chroma_dir=DATA / 'index/chroma_v0').as_retriever(search_kwargs={'k': K_RETRIEVER})

hybride_retriever = build_hybride_retriever([bm25_retriever, chroma_retriever], weights_rrf)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

# llm

In [ ]:
from langchain_core.rate_limiters import InMemoryRateLimiter
limiter = InMemoryRateLimiter(requests_per_second=0.05)
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0,
               reasoning_effort="low",
               rate_limiter=limiter, max_retries=2)

In [18]:
question = "Quel est le ratio CET1 de BNP Paribas ?"
docs = hybride_retriever.invoke(question)[:5]

In [5]:
def format_context(docs):
    """Chunks en texte formate pour llm"""
    blocs = []
    for i, doc in enumerate(docs, start=1):
        m = doc.metadata
        blocs.append(f"[{i}] ({m['banque']}, {m['source']}, page {m['page']})\n{doc.page_content}")
    return "\n\n".join(blocs)

In [ ]:
pp(format_context(docs))

In [48]:
SYSTEM_PROMPT = """Tu es un assistant d'analyse financière. Tu réponds à des questions sur les rapports annuels 2025 de banques, à partir d'extraits numérotés.

Règles :
1. Utilise UNIQUEMENT les informations des extraits. N'utilise jamais tes connaissances propres.
2. Après chaque affirmation, cite le ou les extraits utilisés avec leur numéro entre crochets, par exemple [1] ou [2][4].
3. Si les extraits ne permettent pas de répondre, réponds exactement : « Je ne trouve pas cette information dans les extraits fournis. »
4. Donne les chiffres exactement comme dans les extraits, avec leur unité et leur date.
5. Si la question est ambiguë (par exemple, la banque n'est pas précisée) et que les extraits concernent plusieurs banques, donne la réponse pour chacune, en le précisant.
6. Réponds dans la langue de la question : question en anglais → réponse entièrement en anglais ; question en français → réponse entièrement en français. Le tout de façon concise.
7. Si les extraits portent sur plusieurs entités (groupe consolidé, filiale, maison mère), précise toujours l'entité concernée. Pour une question sur « la banque » sans précision, privilégie le chiffre du groupe consolidé s'il figure dans les extraits."""

USER_TEMPLATE = """Extraits :

{context}

Question : {question}"""

In [16]:
from langchain_core.rate_limiters import InMemoryRateLimiter
limiter = InMemoryRateLimiter(requests_per_second=0.05)
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0,
               reasoning_effort="low",
               rate_limiter=limiter, max_retries=2)

In [49]:
#from src.config import SYSTEM_PROMPT,USER_TEMPLATE

def generate(question, docs, llm) -> str:
    sys_msg = SystemMessage(content=SYSTEM_PROMPT)
    context=format_context(docs)
    hum_msg = HumanMessage(content=USER_TEMPLATE.format(question=question, context=context))
    ai_msg = llm.invoke([sys_msg, hum_msg])
    return ai_msg.content

In [21]:
generate(question,docs,llm)

'Le ratio CET1 de BNP\u202fParibas s’élève à **12,61\u202f%** au 31\u202fdécembre\u202f2025【1】.'

In [23]:
import re

In [40]:
def extraire_citations(reponse, docs):
    sources = re.findall(r"[\[【](\d+)[\]】]",reponse)
    details = []
    vus = set()
    for i in map(int,sources):
        if i in vus or not 1 <= i<= len(docs):
            continue
        vus.add(i)
        m = docs[i-1].metadata
        detail = {'n':i,
                  "banque":m['banque'],
                  "source":m['source'],
                  "annee":m['annee'],
                  "langue":m['langue'],
                  "page":m['page']
                 }
        details.append(detail)
    return details

In [26]:
docs[0].metadata

{'source': 'bnpparibas_2025.pdf',
 'banque': 'BNP Paribas',
 'annee': 2025,
 'langue': 'fr',
 'page': 401,
 'chunk_id': 'bnpparibas_2025_p401_c3'}

In [32]:
rep = 'Le ratio CET1 de BNP\u202fParibas s’élève à **12,61\u202f%** au 31\u202fdécembre\u202f2025【1】.'

In [38]:
extraire_citations(rep,docs)

[{'n': 1,
  'banque': 'BNP Paribas',
  'source': 'bnpparibas_2025.pdf',
  'annee': 2025,
  'langue': 'fr',
  'page': 401}]

In [41]:
extraire_citations("a [2] b 【2】 c [9] d [0]", docs)   # → seulement n=2, une fois

[{'n': 2,
  'banque': 'BNP Paribas',
  'source': 'bnpparibas_2025.pdf',
  'annee': 2025,
  'langue': 'fr',
  'page': 67}]

In [43]:
def answer(question, retriever, llm, k=5) -> dict:
    docs = retriever.invoke(question)[:k]
    reponse = generate(question, docs, llm)
    ans = {"question": question,
           "reponse":reponse,
           "sources": extraire_citations(reponse,docs)}
    return ans

In [51]:
question_list = ['Quel est le ratio CET1 ?', 
                 "Quel est le chiffre d'affaires de la Société Générale ?",
                 "Quel est le chiffre d'affaires de la Société Générale ?"]
ans_list= []
for q in question_list:
    ans = answer(q, hybride_retriever, llm)
    ans_list.append(ans)

In [52]:
ans_list

[{'question': 'Quel est le ratio CET1 ?',
  'reponse': '- **UBS (groupe consolidé)**\u202f: le ratio CET1 était de **14,4\u202f%** au 31\u202fdécembre\u202f2025【3】.  \n- **BNP\u202fParibas**\u202f: le ratio CET1 était de **12,61\u202f%** au 31\u202fdécembre\u202f2025【5】.',
  'sources': [{'n': 3,
    'banque': 'UBS',
    'source': 'ubs_2025.pdf',
    'annee': 2025,
    'langue': 'en',
    'page': 69},
   {'n': 5,
    'banque': 'BNP Paribas',
    'source': 'bnpparibas_2025.pdf',
    'annee': 2025,
    'langue': 'fr',
    'page': 401}]},
 {'question': "Quel est le chiffre d'affaires de la Société Générale ?",
  'reponse': 'Je ne trouve pas cette information dans les extraits fournis.',
  'sources': []},
 {'question': "Quel est le chiffre d'affaires de la Société Générale ?",
  'reponse': 'Je ne trouve pas cette information dans les extraits fournis.',
  'sources': []}]

In [53]:
r = answer("What was UBS's CET1 capital ratio at the end of 2025?", hybride_retriever, llm)
r

{'question': "What was UBS's CET1 capital ratio at the end of 2025?",
 'reponse': 'UBS\u202fAG’s Common Equity Tier\u202f1 (CET1) capital ratio was **14.2\u202f%** at 31\u202fDecember\u202f2025【1】.',
 'sources': [{'n': 1,
   'banque': 'UBS',
   'source': 'ubs_2025.pdf',
   'annee': 2025,
   'langue': 'en',
   'page': 46}]}